In [5]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document


* **FAISS**  Vector store for dense retrieval (semantic search via embeddings).

* **HuggingFaceEmbeddings**
Converts text into embeddings (dense vectors) using a SentenceTransformer model.

* **BM25Retriever**
Sparse retriever (keyword-based) using BM25 scoring.

* **EnsembleRetriever**
Combines multiple retrievers (dense + sparse) into a hybrid retriever using weights.

* **Document**
Standard LangChain document object: page_content (+ optional metadata).

## Example 1 

In [6]:
# Step 1: Sample documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

# Step 2: Dense Retriever (FAISS + HuggingFace)
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever()

In [7]:
### Sparse Retriever(BM25)
sparse_retriever=BM25Retriever.from_documents(docs)
sparse_retriever.k=3 ##top- k documents to retriever

## step 4 : Combine with Ensemble Retriever
hybrid_retriever=EnsembleRetriever(
    retrievers=[dense_retriever,sparse_retriever],
    weight=[0.7,0.3]
)

hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001EAA1BF8620>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001EAA18D4A40>, k=3)], weights=[0.5, 0.5])

In [8]:
# Step 5: Query and get results
query = "How can I build an application using LLMs?"
results = hybrid_retriever.invoke(query)

# Step 6: Print results
for i, doc in enumerate(results):
    print(f"\n🔹 Document {i+1}:\n{doc.page_content}")


🔹 Document 1:
LangChain helps build LLM applications.

🔹 Document 2:
Langchain can be used to develop agentic ai application.

🔹 Document 3:
Langchain has many types of retrievers.

🔹 Document 4:
Pinecone is a vector database for semantic search.


## Example 2:

In [13]:
# 1) Create documents
docs = [
    Document(page_content="ORA-00942 happens when a table or view does not exist or privileges are missing." , metadata= {"doc" : "sample doc"}),
    Document(page_content="OLFM report balance mismatch can occur due to timing differences in postings.", metadata= {"doc" : "sample doc"}),
    Document(page_content="Customer balance mismatch may occur due to currency conversion rounding.", metadata= {"doc" : "sample doc"}),
    Document(page_content="AMS 8.3 report issues are sometimes caused by incorrect joins in mapping logic.", metadata= {"doc" : "sample doc"}),
]
docs

# 2)  Dense retriever (FAISS + embeddings)

embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs,embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k":3})

# 3) Sparse retriever (BM25)

sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k=3

# 4) Hybrid retriever (Ensemble)

hybrid_retriever = EnsembleRetriever(
        retrievers= [dense_retriever,sparse_retriever],
        weights= [0.6,0.4]  # more weight to dense; adjust as needed
    )


# 5) Query
query = "Why am I getting ORA-00942 in AMS 8.3 report?"
results = hybrid_retriever.invoke(query)


for i, d in enumerate(results, 1):
    print(f"{i}. {d.page_content}")

1. AMS 8.3 report issues are sometimes caused by incorrect joins in mapping logic.
2. ORA-00942 happens when a table or view does not exist or privileges are missing.
3. OLFM report balance mismatch can occur due to timing differences in postings.
4. Customer balance mismatch may occur due to currency conversion rounding.


In the example above you provided when we query for # 5) Query query = "Why am I getting ORA-00942 in AMS 8.3 report?" results = hybrid_retriever.invoke(query) for i, d in enumerate(results, 1): print(f"{i}. {d.page_content}") The result we are getting all 4 1. AMS 8.3 report issues are sometimes caused by incorrect joins in mapping logic. 2. ORA-00942 happens when a table or view does not exist or privileges are missing. 3. OLFM report balance mismatch can occur due to timing differences in postings. 4. Customer balance mismatch may occur due to currency conversion rounding. Ideally I should get only the first one right 1. AMS 8.3 report issues are sometimes caused by incorrect joins in mapping logic.

In [14]:
# 1) Set k=1 (the simplest, most direct)

# Dense (FAISS) k=1 + Sparse (BM25) k=1 + Final top_k=1

# Dense retriever
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

# Sparse retriever
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k = 1

# Hybrid
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.6, 0.4],
)

results = hybrid_retriever.invoke("Why am I getting ORA-00942 in AMS 8.3 report?")
print(results[0].page_content)



AMS 8.3 report issues are sometimes caused by incorrect joins in mapping logic.


In [16]:
# 2) Use metadata filters (best practice in enterprise RAG)


docs = [
    Document(page_content="ORA-00942 happens when a table or view does not exist or privileges are missing." ,metadata={"report": "oracle", "topic": "error", "code": "ORA-00942"}),
    Document(page_content="OLFM report balance mismatch can occur due to timing differences in postings.",  metadata={"report": "AMS 8.3", "topic": "report_issue"}),
    Document(page_content="Customer balance mismatch may occur due to currency conversion rounding.", metadata={"report": "OLFM"}),
    Document(page_content="AMS 8.3 report issues are sometimes caused by incorrect joins in mapping logic.", metadata={"report": "general"}),
]
docs 



query = "Why am I getting ORA-00942 in AMS 8.3 report?"
results = hybrid_retriever.invoke(query)

filtered = [d for d in results if d.metadata.get("report") == "AMS 8.3"]
print(filtered[0].page_content if filtered else results[0].page_content)


AMS 8.3 report issues are sometimes caused by incorrect joins in mapping logic.


In [17]:
# Rerank and keep only the best (best quality)

def score_boost(doc, query):
    boost = 0
    if "AMS 8.3" in query and doc.metadata.get("report") == "AMS 8.3":
        boost += 10
    if "ORA-" in query and "ORA-" in doc.page_content:
        boost += 5
    return boost

results = hybrid_retriever.invoke(query)
results = sorted(results, key=lambda d: score_boost(d, query), reverse=True)

print(results[0].page_content)


AMS 8.3 report issues are sometimes caused by incorrect joins in mapping logic.


In [18]:
#  Quick fix for your exact example (without changing your dataset much)

query = "Why am I getting ORA-00942 in AMS 8.3 report?"
results = hybrid_retriever.invoke(query)

# Prefer docs mentioning "AMS 8.3" if query mentions it
preferred = [d for d in results if "AMS 8.3" in d.page_content]

final = preferred[0] if preferred else results[0]
print(final.page_content)


AMS 8.3 report issues are sometimes caused by incorrect joins in mapping logic.


### It's not about only AMS 8.3 in case if query why "customer balance is wrong" then also it should query the exact match?

Why you’re getting multiple docs

Hybrid retrieval is designed to maximize recall (bring in useful context). With small corpora (or broad queries), many docs look “somewhat relevant”, so you get multiple.

To make it behave like “exact match”, you need precision controls:

top_k = 1 at the end

thresholding (drop weak matches)

precision-first scoring (usually sparse/BM25 dominates when you care about exact text)

Below are 3 practical patterns that work well.

In [19]:
# Pattern A: “Top 1 Only” + Strong Sparse Weight (Simple & Effective)
"""
If your goal is “return the single best matching chunk”, do:

Dense k = 5 (let semantic help a bit)

Sparse k = 5 (exact match candidates)

Final: return top 1

Weight: sparse heavier (because you said “exact match”)

"""

dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k = 5

hybrid = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.3, 0.7],   # sparse dominates for “exact match”
)

best_doc = hybrid.invoke("customer balance is wrong")[0]
print(best_doc.page_content)


AMS 8.3 report issues are sometimes caused by incorrect joins in mapping logic.


In [20]:
import re
from typing import List
from langchain_core.documents import Document

def tokens(s: str) -> set:
    return set(re.findall(r"[a-z0-9]+", s.lower()))

def best_exact_doc(query: str, docs: List[Document], min_overlap: int = 2) -> Document:
    q = tokens(query)

    scored = []
    for d in docs:
        d_tokens = tokens(d.page_content)
        overlap = len(q & d_tokens)
        scored.append((overlap, d))

    # Sort by highest overlap
    scored.sort(key=lambda x: x[0], reverse=True)

    # If no doc meets overlap threshold, still return best one (or you can return None)
    if scored[0][0] < min_overlap:
        # Option 1: return best anyway
        return scored[0][1]
        # Option 2 (stricter): return None

    return scored[0][1]

# --- usage ---
query = "customer balance is wrong"
candidates = hybrid_retriever.invoke(query)   # may return multiple
best = best_exact_doc(query, candidates, min_overlap=2)

print(best.page_content)


Customer balance mismatch may occur due to currency conversion rounding.


In [21]:
def boosted_exact_score(query: str, doc_text: str) -> int:
    q = tokens(query)
    d = tokens(doc_text)

    overlap = len(q & d)

    # Boost for patterns like ORA-00942, IDs, versions
    if "ora" in q and "ora" in d:
        overlap += 3
    if any(t.isdigit() for t in q) and any(t.isdigit() for t in d):
        overlap += 1

    return overlap

def best_doc_with_boost(query: str, docs: List[Document]) -> Document:
    scored = [(boosted_exact_score(query, d.page_content), d) for d in docs]
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1]

query = "Why am I getting ORA-00942 in AMS 8.3 report?"
candidates = hybrid_retriever.invoke(query)
best = best_doc_with_boost(query, candidates)
print(best.page_content)


AMS 8.3 report issues are sometimes caused by incorrect joins in mapping logic.


In [22]:
# Pattern B: Add a Precision Gate (Drop Weak Matches)
""" You want: “only return a doc if it truly matches; otherwise don’t return noise.”

Since EnsembleRetriever often doesn’t expose scores cleanly, a practical gate is term overlap (cheap but effective), especially for “exact match” behavior. """ 


import re

def token_set(text: str):
    return set(re.findall(r"[a-z0-9]+", text.lower()))

def exactness_gate(query: str, doc_text: str, min_overlap=2):
    q = token_set(query)
    d = token_set(doc_text)
    overlap = len(q & d)
    return overlap >= min_overlap, overlap

query = "customer balance is wrong"
candidates = hybrid.invoke(query)   # returns a list
scored = []
for d in candidates:
    ok, overlap = exactness_gate(query, d.page_content, min_overlap=2)
    if ok:
        scored.append((overlap, d))

if not scored:
    print("No strong exact match found (avoid returning noise).")
else:
    scored.sort(reverse=True, key=lambda x: x[0])
    best = scored[0][1]
    print(best.page_content)


Customer balance mismatch may occur due to currency conversion rounding.


In [25]:
#  Pattern C: True “Exact Match Mode” Using a Two-Stage Router (Best Practice)

""" This is the most reliable production pattern:

Try sparse-only first (BM25 exactness)

If it’s too weak (or no overlap), fall back to dense

It behaves exactly like: “prefer exact match when available. """


def retrieve_exact_first(query: str, top_k=1, min_overlap=2):
    # Stage 1: sparse (exact)
    sparse_docs = sparse_retriever.invoke(query)
    sparse_docs = sparse_docs[:5]

    # Keep only sparse docs with enough overlap
    strong = []
    for d in sparse_docs:
        ok, overlap = exactness_gate(query, d.page_content, min_overlap=min_overlap)
        if ok:
            strong.append((overlap, d))

    if strong:
        strong.sort(reverse=True, key=lambda x: x[0])
        return [strong[0][1]]  # top 1 exact match

    # Stage 2: dense fallback (meaning)
    dense_docs = dense_retriever.invoke(query)
    return dense_docs[:top_k]

best = retrieve_exact_first("customer balance is wrong", top_k=1, min_overlap=2)[0]
print(best.page_content)


Customer balance mismatch may occur due to currency conversion rounding.
